## Sprints Data transform Bronze to Silver

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01-Environment_Config

In [0]:
%run ../00-common/03-Silver_Helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.sprints"
silver_table = f"{catalog_name}.{silver_schema}.sprints"

In [0]:
from pyspark.sql import functions as f

In [0]:
# To use reader api options 
sprints_df = spark.read.table(bronze_table).filter((f.col("batch_id") == v_batch_id))

# sprints_df = spark.table(bronze_table) - without any reader options
# Both returns the same results

In [0]:
column_rename = {
    "date" : "race_date",
    "grid" : "grid_position",
    "laps" : "completed_laps",
    "number" : "car_number",
    "position" : "finish_position",
    "constructorId" : "constructor_id",
    "driverId" : "driver_id",
    "raceName" : "race_name",
    "positionText" : "finish_position_text"
    }

sprints_renamed_df = sprints_df.drop(f.col("url"))\
                                .withColumnsRenamed(column_rename)

In [0]:
# To Drop duplicates and Null values in a column - distinct(), dropDuplicates() & filter()

sprints_filtered_df = sprints_renamed_df.filter(
    (
        (f.col("driver_id").isNotNull()) & (f.col("constructor_id").isNotNull()) & (f.col("round").isNotNull()) & (f.col("season").isNotNull())
    )
).dropDuplicates(["driver_id","constructor_id","round","season"])

In [0]:
sprints_final_df = sprints_filtered_df.withColumn("race_name", f.initcap("race_name"))

In [0]:
columns = [
    "race_name",
    "race_date",
    "grid_position",
    "completed_laps",
    "car_number",
    "points",
    "finish_position",
    "finish_position_text",
    "status",
    "ingestion_timestamp",
    "source_file",
    "batch_id"
]

write_to_silver(sprints_final_df, silver_table, "trg.season = src.season AND trg.round = src.round AND trg.constructor_id = src.constructor_id AND trg.driver_id = src.driver_id", columns)